# Transport and emission output

Both writers create one file per shape plus a manifest. This tutorial writes a
transport table, reads the manifest, resumes it, then writes an emission result.


In [ ]:
import tempfile
from pathlib import Path

import pandas as pd

from gtfs2emis_py import emission_model, transport_model
from gtfs2emis_py.emission_writer import read_emission_manifest, write_emission_output
from gtfs2emis_py.gtfs.writer import read_transport_manifest

DATA = Path.cwd() / "tests/data"
tmp = tempfile.TemporaryDirectory()
output = Path(tmp.name)
manifest = transport_model(DATA / "synthetic_edge_cases.zip", parallel=False, output_path=output)
print("transport shapes:", manifest.counts["shapes"], "rows:", manifest.counts["rows"])
print("files:", sorted(path.name for path in (output / "shapes").glob("*.parquet")))

In [ ]:
on_disk = read_transport_manifest(output)
print(
    {
        key: on_disk[key]
        for key in ("schema_version", "rows_total", "rows_written", "shapes_resumed")
    }
)
resumed = transport_model(
    DATA / "synthetic_edge_cases.zip", parallel=False, output_path=output, resume=True
)
print("resumed shapes:", resumed.counts["shapes"])
print("parameters:", resumed.parameters)

In [ ]:
tp = transport_model(DATA / "synthetic_edge_cases.zip", parallel=False)
fleet = pd.DataFrame(
    {
        "type_name_br": ["BUS_URBAN_D"],
        "model_year": [2015],
        "fuel": ["D"],
        "fleet_composition": [1.0],
    }
)
result = emission_model(tp, "ef_brazil_cetesb", fleet, ["CO", "NOx"])
emission_dir = Path(tmp.name) / "emissions"
written = write_emission_output(result, emission_dir)
emission_manifest = read_emission_manifest(emission_dir)
print("emission rows:", emission_manifest["rows"], "shapes:", len(emission_manifest["shapes"]))
pd.read_parquet(emission_dir / emission_manifest["shapes"][0]["file"]).head()

A failed shape is recorded as `failed` while successful files and the
manifest remain available before `TransportWriteError` is raised. Emission
resume also includes a value hash, so changed values are rewritten rather than
reusing stale files.
